# Uncertainty Quantification for ECG Forecasting

Techniques covered:
1. **Monte Carlo Dropout** — Bayesian approximation of model uncertainty
2. **Epistemic vs Aleatoric** uncertainty decomposition
3. **Prediction interval coverage** analysis
4. **Per-lead calibration** — is uncertainty aligned with actual error?

## 1. Setup and Imports

In [ ]:
import os
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

## 2. Reproducibility and Device

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 3. Paths and Constants

In [ ]:
SAVE_DIR  = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'reports', 'checkpoints')
FIG_DIR   = os.path.join('..', 'reports', 'figures', 'uncertainty')
os.makedirs(FIG_DIR, exist_ok=True)

FS         = 100
INPUT_LEN  = 500
HORIZON    = 100
N_LEADS    = 12
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

COLORS = {
    'bg':      '#ffffff',
    'grid':    '#e0e0e0',
    'text':    '#2c3e50',
    'accent1': '#3498db',
    'accent2': '#e74c3c',
    'accent3': '#2ecc71',
}

plt.rcParams.update({
    'figure.facecolor': COLORS['bg'],
    'axes.facecolor':   COLORS['bg'],
    'figure.dpi':       120,
    'savefig.dpi':      150,
})

print(f"Input: {INPUT_LEN/FS:.1f}s  |  Horizon: {HORIZON/FS:.1f}s  |  Leads: {N_LEADS}")

## 4. Load Data

In [ ]:
X_test = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

n_samples_uq = min(50, len(X_test))
X_uq = X_test[:n_samples_uq]
y_uq = y_test[:n_samples_uq]

X_uq_torch = torch.tensor(X_uq, dtype=torch.float32).to(DEVICE)
y_uq_torch = torch.tensor(y_uq, dtype=torch.float32).to(DEVICE)

print(f"X_uq: {X_uq.shape}")
print(f"y_uq: {y_uq.shape}")

## 5. Model Definition

CNN-LSTM with MC Dropout support. The `enable_mc_dropout()` method keeps dropout active during inference so each forward pass samples different weights.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, pool=True):
        super().__init__()
        ops = [
            nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=k // 2, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
        ]
        if pool:
            ops.append(nn.MaxPool1d(2))
        self.net = nn.Sequential(*ops)

    def forward(self, x):
        return self.net(x)


class CNNLSTMForecaster(nn.Module):
    def __init__(self, n_leads=12, horizon=100, dropout=0.3,
                 lstm_hidden=128, bidirectional=True):
        super().__init__()
        self.horizon      = horizon
        self.n_leads      = n_leads
        self.D            = 2 if bidirectional else 1
        self.dropout_rate = dropout

        self.cnn = nn.Sequential(
            ConvBlock(n_leads, 32,  k=7, pool=True),
            ConvBlock(32,      64,  k=5, pool=True),
            ConvBlock(64,      128, k=3, pool=True),
            ConvBlock(128,     128, k=3, pool=False),
        )
        self.lstm = nn.LSTM(
            input_size=128, hidden_size=lstm_hidden, num_layers=2,
            batch_first=True, dropout=dropout, bidirectional=bidirectional,
        )
        self.attn = nn.Sequential(
            nn.Linear(lstm_hidden * self.D, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.decoder = nn.Sequential(
            nn.LayerNorm(lstm_hidden * self.D),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * self.D, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, horizon * n_leads),
        )

    def forward(self, x):
        f      = self.cnn(x).permute(0, 2, 1)
        enc, _ = self.lstm(f)
        w      = torch.softmax(self.attn(enc), dim=1)
        ctx    = (w * enc).sum(dim=1)
        out    = self.decoder(ctx)
        return out.view(-1, self.horizon, self.n_leads)

    def enable_mc_dropout(self):
        """Put model in eval mode but keep all Dropout layers in train mode."""
        self.eval()
        for module in self.modules():
            if isinstance(module, nn.Dropout):
                module.train()

## 6. Load Trained Model

In [ ]:
model = CNNLSTMForecaster(n_leads=N_LEADS, horizon=HORIZON, dropout=0.3).to(DEVICE)

model_path = os.path.join(MODEL_DIR, 'CNN-LSTM_final.pt')
if os.path.exists(model_path):
    # weights_only=True required for PyTorch >= 2.4
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=True)
    state_dict = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model.load_state_dict(state_dict)
    print("Model loaded successfully")
else:
    print("No checkpoint found -- using untrained model (results will be random)")

model.eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Monte Carlo Dropout Predictions

Run 50 stochastic forward passes with dropout active.
The variance across passes estimates **epistemic uncertainty** (model uncertainty).
The mean gives the best point prediction.

In [ ]:
def mc_dropout_predictions(model, X, n_iterations=50):
    """
    Run n_iterations stochastic forward passes.

    Returns
    -------
    predictions : ndarray  shape (n_iterations, N, horizon, n_leads)
    """
    model.enable_mc_dropout()   # eval + dropout layers in train mode
    predictions = []
    with torch.no_grad():
        for _ in tqdm(range(n_iterations), desc='MC Dropout'):
            pred = model(X)
            predictions.append(pred.cpu().numpy())
    return np.array(predictions)


print("Generating MC Dropout predictions (50 iterations) ...")
mc_predictions = mc_dropout_predictions(model, X_uq_torch, n_iterations=50)

mc_mean = mc_predictions.mean(axis=0)   # (N, horizon, n_leads)
mc_std  = mc_predictions.std(axis=0)    # (N, horizon, n_leads)

print(f"MC predictions : {mc_predictions.shape}")
print(f"MC mean        : {mc_mean.shape}")
print(f"MC std         : {mc_std.shape}")

## 8. Uncertainty Bands per Lead

Shaded regions show 68% (±1σ) and 95% (±2σ) prediction intervals for the first test sample.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for idx, lead_idx in enumerate([0, 1, 2, 6, 10, 11]):
    ax        = axes[idx]
    t         = np.arange(HORIZON) / FS
    y_mean    = mc_mean[0, :, lead_idx]
    y_std     = mc_std[0,  :, lead_idx]
    y_true    = y_uq[0,    :, lead_idx]

    ax.fill_between(t, y_mean - 2*y_std, y_mean + 2*y_std,
                    alpha=0.3, color=COLORS['accent1'], label='95% CI')
    ax.fill_between(t, y_mean - y_std,   y_mean + y_std,
                    alpha=0.5, color=COLORS['accent1'], label='68% CI')
    ax.plot(t, y_mean, color=COLORS['accent2'], lw=2, label='Mean prediction')
    ax.plot(t, y_true, color=COLORS['accent3'], lw=2, label='Ground truth')

    ax.set_title(f'Lead {LEAD_NAMES[lead_idx]} - MC Dropout Uncertainty',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (mV)')
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_mc_dropout_uncertainty.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_mc_dropout_uncertainty.png")

## 9. Epistemic vs Aleatoric Uncertainty

- **Epistemic** (model uncertainty): captured by the variance across MC Dropout passes — reducible with more data/training.
- **Aleatoric** (data uncertainty): the irreducible noise measured by mean absolute prediction error.

In [ ]:
epistemic_unc = float(mc_std.mean())
aleatoric_unc = float(np.mean(np.abs(mc_mean - y_uq)))

print("=" * 60)
print("  UNCERTAINTY DECOMPOSITION")
print("=" * 60)
print(f"  Epistemic (model) : {epistemic_unc:.4f}  (reducible with more data)")
print(f"  Aleatoric (data)  : {aleatoric_unc:.4f}  (irreducible noise)")

## 10. Uncertainty Heatmap (Lead x Time)

In [ ]:
mc_std_mean = mc_std.mean(axis=0)          # avg over samples -> (horizon, n_leads)
error_mean  = np.abs(mc_mean - y_uq).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(mc_std_mean.T, aspect='auto', cmap='YlOrRd')
axes[0].set_xlabel('Forecast Time Step')
axes[0].set_ylabel('Lead')
axes[0].set_yticks(range(N_LEADS))
axes[0].set_yticklabels(LEAD_NAMES, fontsize=9)
axes[0].set_title('MC Dropout Uncertainty (Epistemic)', fontweight='bold')
plt.colorbar(im1, ax=axes[0], label='Std Dev')

im2 = axes[1].imshow(error_mean.T, aspect='auto', cmap='RdYlBu_r')
axes[1].set_xlabel('Forecast Time Step')
axes[1].set_ylabel('Lead')
axes[1].set_yticks(range(N_LEADS))
axes[1].set_yticklabels(LEAD_NAMES, fontsize=9)
axes[1].set_title('Prediction Error (Aleatoric proxy)', fontweight='bold')
plt.colorbar(im2, ax=axes[1], label='Mean |Error|')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_uncertainty_decomposition.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_uncertainty_decomposition.png")

## 11. Prediction Interval Coverage Analysis

For a well-calibrated model, a 90% prediction interval should contain ~90% of actual values.

In [ ]:
lower_bound      = mc_mean - 1.645 * mc_std
upper_bound      = mc_mean + 1.645 * mc_std
in_bounds        = (y_uq >= lower_bound) & (y_uq <= upper_bound)
coverage_per_step = in_bounds.mean(axis=(0, 2))   # avg over samples & leads -> (horizon,)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(range(HORIZON), coverage_per_step * 100, 'o-', lw=2, markersize=6,
        color=COLORS['accent1'], label='Empirical Coverage')
ax.axhline(90, color=COLORS['accent2'], ls='--', lw=2, label='Target 90%')
ax.fill_between(range(HORIZON), 85, 95, alpha=0.2, color=COLORS['accent3'],
                label='Acceptable band (85-95%)')
ax.set_xlabel('Forecast Time Step')
ax.set_ylabel('Coverage (%)')
ax.set_title('Prediction Interval Coverage Analysis (90% CI)', fontweight='bold', fontsize=12)
ax.set_ylim([0, 105])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_coverage_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: 03_coverage_analysis.png")
print(f"Average empirical coverage: {coverage_per_step.mean()*100:.1f}%")

## 12. Per-Lead Uncertainty and Calibration

Compares MC uncertainty (std) with actual prediction error per lead. A well-calibrated model has uncertainty proportional to its error.

In [ ]:
mc_std_per_lead = mc_std.mean(axis=(0, 1))                    # (n_leads,)
error_per_lead  = np.abs(mc_mean - y_uq).mean(axis=(0, 1))   # (n_leads,)

x_pos = np.arange(N_LEADS)
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar: uncertainty vs error per lead
axes[0].bar(x_pos - width/2, mc_std_per_lead, width,
            label='MC Uncertainty (Std)', color=COLORS['accent1'], alpha=0.8)
axes[0].bar(x_pos + width/2, error_per_lead,  width,
            label='Prediction Error',     color=COLORS['accent2'], alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(LEAD_NAMES, fontsize=11)
axes[0].set_ylabel('Value (mV)')
axes[0].set_title('Uncertainty & Error by Lead', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Calibration scatter
axes[1].scatter(mc_std_per_lead, error_per_lead, s=150, alpha=0.7,
                color=COLORS['accent3'], edgecolor='black', linewidth=1.5)
for i, lead in enumerate(LEAD_NAMES):
    axes[1].annotate(lead, (mc_std_per_lead[i], error_per_lead[i]),
                     fontsize=9, ha='center', va='bottom')
max_val = max(mc_std_per_lead.max(), error_per_lead.max()) * 1.1
axes[1].plot([0, max_val], [0, max_val], 'k--', lw=1, alpha=0.5, label='Perfect calibration')
axes[1].set_xlabel('MC Uncertainty (Std)')
axes[1].set_ylabel('Prediction Error')
axes[1].set_title('Calibration: Uncertainty vs Error', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_uncertainty_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 04_uncertainty_metrics.png")

## 13. Summary Statistics

In [ ]:
print("\n" + "=" * 60)
print("  UNCERTAINTY QUANTIFICATION SUMMARY")
print("=" * 60)
print("\n  Monte Carlo Dropout Settings:")
print("    Iterations         : 50")
print("    Confidence intervals: 68% (+/-1 sigma), 95% (+/-2 sigma)")
print("\n  Key Findings:")
print(f"    Mean uncertainty   : {mc_std.mean():.4f} mV")
print(f"    Mean error         : {error_per_lead.mean():.4f} mV")
print(f"    Coverage (90% CI)  : {coverage_per_step.mean()*100:.1f}%")
print(f"    Epistemic unc      : {epistemic_unc:.4f}")
print(f"    Aleatoric unc      : {aleatoric_unc:.4f}")
print("\n  Lead-level results:")
print(f"    Highest uncertainty: {LEAD_NAMES[np.argmax(mc_std_per_lead)]:>5s}"
      f"  (std={mc_std_per_lead.max():.4f} mV)")
print(f"    Most predictable   : {LEAD_NAMES[np.argmin(error_per_lead)]:>5s}"
      f"  (err={error_per_lead.min():.4f} mV)")
print("\nUncertainty Quantification Complete")

## 14. Save Results

In [ ]:
results = {
    'mc_mean':              mc_mean,
    'mc_std':               mc_std,
    'epistemic_unc':        epistemic_unc,
    'aleatoric_unc':        aleatoric_unc,
    'coverage_90':          float(coverage_per_step.mean()),
    'uncertainty_per_lead': mc_std_per_lead.tolist(),
    'error_per_lead':       error_per_lead.tolist(),
}

out_path = os.path.join(FIG_DIR, 'uncertainty_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to: {out_path}")
print(f"Figures saved to: {os.path.abspath(FIG_DIR)}")